# Module 07 — ReAct with no SDK support: you write the regex

**THE ONE IDEA:** before native tool calling existed, an agent was a **prompt asking for
a text format** plus a **regex you wrote** to claw the tool call back out of prose.

The model now chooses the sequence. That is the line between Block B and Block C.

Run it twice — once on `gpt-4.1-mini`, once on the **1B** Llama. The 1B is here on
purpose. When the parser misses, you are watching **FM1 — tools-as-text**, the first of
the five failure modes, live rather than described.


In [1]:
import sys; sys.path.insert(0, "..")   # shared _providers.py / _tools.py at the phase root
import re
from _providers import get_client
from _tools import run_tool

SYSTEM = """Answer the question using these tools:
  search_policy(query)  - look up bank policy by keyword
  calculate(expression) - evaluate arithmetic

Reply in EXACTLY this format, one step at a time:
Thought: <reasoning>
Action: <tool_name>
Action Input: <argument>

When you have the answer:
Thought: <reasoning>
Final Answer: <answer>"""

QUESTION = ("What is the early repayment charge in year 2 on a 250000 loan? "
            "Look up the policy, then calculate it.")

ACTION_RE = re.compile(r"Action:\s*(\w+)\s*\n\s*Action Input:\s*(.+)", re.I)
FINAL_RE  = re.compile(r"Final Answer:\s*(.+)", re.I | re.S)
print("the parser IS the integration — two regexes, hand written")

the parser IS the integration — two regexes, hand written


## The loop

Note what does the work: `ACTION_RE.search(text)`. There is no `tool_calls` field. The
model wrote prose and we are mining it.

In [2]:
def react(provider, max_steps=5, verbose=True):
    client, model, _ = get_client(provider)
    transcript = f"{SYSTEM}\n\nQuestion: {QUESTION}\n"
    for step in range(1, max_steps + 1):
        r = client.chat.completions.create(
            model=model, max_tokens=300, stop=["Observation:"],
            messages=[{"role": "user", "content": transcript}])
        text = r.choices[0].message.content or ""
        if verbose: print(f"\n--- step {step} ---\n{text.strip()[:280]}")

        # ORDER MATTERS. A rambling model emits an Action AND a Final Answer in the
        # same turn. Checking FINAL_RE first accepts an answer computed from NOTHING —
        # the first run of this notebook returned the literal '[calculated amount]'
        # and reported ok=True. An Action always wins over a Final Answer beside it.
        act, fin = ACTION_RE.search(text), FINAL_RE.search(text)
        if fin and not act:
            return fin.group(1).strip(), step, True
        if not (m := act):
            if verbose: print("  !! PARSER MISS — no Action/Action Input found")
            return None, step, False

        name, arg = m.group(1).strip(), m.group(2).strip().strip('"')
        key = "query" if name == "search_policy" else "expression"
        obs = run_tool(name, {key: arg})
        if verbose: print(f"  -> {name}({arg!r})\n  Observation: {obs[:90]}")
        transcript += text + f"\nObservation: {obs}\n"
    return None, max_steps, False

## Run 1 — a model that holds the format

In [3]:
ans, steps, ok = react("openai")
print(f"\n\nRESULT  ok={ok}  steps={steps}\n{ans}")


--- step 1 ---
Thought: I need to look up the bank policy for the early repayment charge by the keyword "early repayment charge year 2" to find the percentage or formula applied to the loan amount in the second year.  
Action: search_policy  
Action Input: "early repayment charge year 2"
  -> search_policy('early repayment charge year 2')
  Observation: No policy matched. Try one of: ltv, erc, income, affordability, rates, tenor, deposit, val

--- step 2 ---
Thought: Since "early repayment charge year 2" did not yield results, I will try the keyword "erc" (early repayment charge) which is a common abbreviation.  
Action: search_policy  
Action Input: "erc"
  -> search_policy('erc')
  Observation: Early repayment charge: 5% yr1, 4% yr2, 3% yr3, 2% yr4, 1% yr5.

--- step 3 ---
Thought: The early repayment charge in year 2 is 4% of the loan amount. I need to calculate 4% of 250,000.  
Action: calculate  
Action Input: 250000 * 0.04
  -> calculate('250000 * 0.04')
  Observation: 10000.0



## Run 2 — the 1B, where the parser breaks

`_providers.py` ships `llama3.2` (3B). Point it at the **1B** for this cell only. A weak
model chats around the format instead of obeying it.

In [4]:
import _providers
_providers.CONFIG["local"]["model"] = "llama3.2:1b"      # local override, this cell only

ans1b, steps1b, ok1b = react("local")
print(f"\n\nRESULT  ok={ok1b}  steps={steps1b}\n{ans1b}")

_providers.CONFIG["local"]["model"] = "llama3.2"         # put it back


--- step 1 ---
Since we don't have any default search policy for looking up bank policies or calculating specific charges, I'll assume you want me to provide the general information and later calculate the exact late fees if you specify them.

Thought: To look up the early repayment charge (ERC
  !! PARSER MISS — no Action/Action Input found


RESULT  ok=False  steps=1
None


## The lesson

In [5]:
print(f"gpt-4.1-mini  parsed={ok}    steps={steps}")
print(f"llama3.2:1b   parsed={ok1b}  steps={steps1b}")
print()
print("LESSON — with no native tool calling, the INTEGRATION IS A REGEX, and its")
print("reliability is the model's willingness to hold a text format.")
print()
print("That is FM1 — tools-as-text. The model 'called' the tool in prose; your")
print("orchestrator could not see it, so nothing ran. No exception was raised.")
print("The agent simply did nothing and reported something.")
print()
print("Two things follow:")
print("  - the weaker the model, the more the format slips (that is why the 1B is here)")
print("  - every prompt token spent teaching the format is a token not spent reasoning")
print()
print("Module 08 deletes both regexes. The format moves into the API contract, where")
print("the provider enforces it instead of you.")

gpt-4.1-mini  parsed=True    steps=4
llama3.2:1b   parsed=False  steps=1

LESSON — with no native tool calling, the INTEGRATION IS A REGEX, and its
reliability is the model's willingness to hold a text format.

That is FM1 — tools-as-text. The model 'called' the tool in prose; your
orchestrator could not see it, so nothing ran. No exception was raised.
The agent simply did nothing and reported something.

Two things follow:
  - the weaker the model, the more the format slips (that is why the 1B is here)
  - every prompt token spent teaching the format is a token not spent reasoning

Module 08 deletes both regexes. The format moves into the API contract, where
the provider enforces it instead of you.


---

**Next:** `08_agent_with_sdk_tool_calling.ipynb` — the regex disappears.